# Задание 1. Код Хаффмана

Берем большой текст из Project Gutenberg: **The Brothers Karamazov**.

In [1]:
import heapq
import itertools
import json
import math
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

## 1. Загружаем текст

In [2]:
LOCAL_TEXT_PATHS = [
    Path("data/brothers_karamazov.txt"),
    Path("../data/brothers_karamazov.txt"),
    Path("brothers_karamazov.txt"),
]

for path in LOCAL_TEXT_PATHS:
    if path.exists():
        TEXT_PATH = path
        break
else:
    raise FileNotFoundError("Не найден brothers_karamazov.txt. Проверь папку data/.")

print("Файл текста:", TEXT_PATH)

raw_text = TEXT_PATH.read_text(encoding="utf-8").replace("\r\n", "\n").replace("\r", "\n")


def remove_gutenberg_wrapper(text):
    start = re.search(r"\*\*\* START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)
    end = re.search(r"\*\*\* END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", text)

    if start:
        text = text[start.end():]
    if end:
        text = text[:end.start()]

    return text.strip()


text = remove_gutenberg_wrapper(raw_text)
original_size = len(text.encode("utf-8"))

print("Символов:", len(text))
print("Байт UTF-8:", original_size)
print(text[:500])

Файл текста: ../data/brothers_karamazov.txt
Символов: 1936072
Байт UTF-8: 1982964
The Brothers Karamazov

Translated from the Russian of

Fyodor Dostoyevsky

by Constance Garnett

The Lowell Press

New York


Contents

 Part I
 Book I. The History Of A Family
 Chapter I. Fyodor Pavlovitch Karamazov
 Chapter II. He Gets Rid Of His Eldest Son
 Chapter III. The Second Marriage And The Second Family
 Chapter IV. The Third Son, Alyosha
 Chapter V. Elders
 Book II. An Unfortunate Gathering
 Chapter I. They Arrive At The Monastery
 Chapter II. The Old Buffoon
 Chapter III. Peasant W


## 2. Реализация Хаффмана

In [3]:
@dataclass
class Node:
    symbol: object = None
    left: object = None
    right: object = None

    def is_leaf(self):
        return self.symbol is not None


def split_blocks(sequence, block_size):
    return [sequence[i:i + block_size] for i in range(0, len(sequence), block_size)]


def build_huffman_tree(frequencies):
    order = itertools.count()
    heap = [(freq, next(order), Node(symbol=symbol)) for symbol, freq in frequencies.items()]
    heapq.heapify(heap)

    if len(heap) == 1:
        return heap[0][2]

    while len(heap) > 1:
        freq_1, _, node_1 = heapq.heappop(heap)
        freq_2, _, node_2 = heapq.heappop(heap)
        parent = Node(left=node_1, right=node_2)
        heapq.heappush(heap, (freq_1 + freq_2, next(order), parent))

    return heap[0][2]


def build_codes(root):
    codes = {}

    def walk(node, prefix):
        if node.is_leaf():
            codes[node.symbol] = prefix or "0"
            return

        walk(node.left, prefix + "0")
        walk(node.right, prefix + "1")

    walk(root, "")
    return codes


def encode_blocks(blocks, codes):
    return "".join(codes[block] for block in blocks)


def decode_bits(bits, root):
    result = []
    node = root

    for bit in bits:
        node = node.left if bit == "0" else node.right

        if node.is_leaf():
            result.append(node.symbol)
            node = root

    return result


def entropy(frequencies):
    total = sum(frequencies.values())
    result = 0

    for freq in frequencies.values():
        p = freq / total
        result -= p * math.log2(p)

    return result


def average_code_length(frequencies, codes):
    total = sum(frequencies.values())
    return sum((freq / total) * len(codes[symbol]) for symbol, freq in frequencies.items())

## 3. Кодируем блоками из 1 и 2 символов

In [4]:
def huffman_experiment(text, block_size):
    blocks = split_blocks(text, block_size)
    frequencies = Counter(blocks)
    tree = build_huffman_tree(frequencies)
    codes = build_codes(tree)

    encoded_bits = encode_blocks(blocks, codes)
    decoded_text = "".join(decode_bits(encoded_bits, tree))

    assert decoded_text == text

    block_entropy = entropy(frequencies)
    block_avg_length = average_code_length(frequencies, codes)
    symbols_per_block = len(text) / len(blocks)

    return {
        "block_size": block_size,
        "alphabet_size": len(frequencies),
        "entropy_per_block": block_entropy,
        "avg_len_per_block": block_avg_length,
        "entropy_per_symbol": block_entropy / symbols_per_block,
        "avg_len_per_symbol": block_avg_length / symbols_per_block,
        "bits": encoded_bits,
        "encoded_bits": len(encoded_bits),
        "compressed_bytes_without_metadata": math.ceil(len(encoded_bits) / 8),
        "compression_ratio_without_metadata": math.ceil(len(encoded_bits) / 8) / original_size,
        "codes": codes,
        "frequencies": frequencies,
    }


results = [huffman_experiment(text, 1), huffman_experiment(text, 2)]

comparison = pd.DataFrame([
    {key: value for key, value in result.items() if key not in ["bits", "codes", "frequencies"]}
    for result in results
])

## 4. Сохраняем промежуточные файлы

Сохраняем сжатый битовый поток и минимальную таблицу декодирования. `.bin` содержит упакованные биты, `.json` содержит только `block_size`, `bit_length` и `codes`. В итоговой таблице ниже размер архива считается как `.bin + .json`.

In [5]:
OUTPUT_DIR = TEXT_PATH.parent
OUTPUT_DIR.mkdir(exist_ok=True)


def pack_bits(bit_string):
    padding = (-len(bit_string)) % 8
    padded = bit_string + "0" * padding

    packed = bytearray()
    for index in range(0, len(padded), 8):
        packed.append(int(padded[index:index + 8], 2))

    return bytes(packed), padding


def save_huffman_package(result):
    prefix = OUTPUT_DIR / f"huffman_block_{result['block_size']}"
    bin_path = prefix.with_suffix(".bin")
    meta_path = prefix.with_suffix(".json")

    packed_bits, padding = pack_bits(result["bits"])
    bin_path.write_bytes(packed_bits)

    meta = {
        "block_size": result["block_size"],
        "bit_length": len(result["bits"]),
        "codes": [
            [block, code]
            for block, code in result["codes"].items()
        ],
    }

    meta_path.write_text(json.dumps(meta, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")

    return bin_path, meta_path


saved_files = []
for result in results:
    bin_path, meta_path = save_huffman_package(result)
    saved_files.append((bin_path, meta_path))

saved_sizes = pd.DataFrame([
    {
        "block_size": result["block_size"],
        "archive_payload_bytes": bin_path.stat().st_size,
        "metadata_bytes": meta_path.stat().st_size,
        "total_bytes": bin_path.stat().st_size + meta_path.stat().st_size,
        "compression_ratio_with_metadata": (bin_path.stat().st_size + meta_path.stat().st_size) / original_size,
    }
    for result, (bin_path, meta_path) in zip(results, saved_files)
])

comparison = comparison.merge(saved_sizes, on="block_size")

columns = [
    "block_size",
    "alphabet_size",
    "entropy_per_symbol",
    "avg_len_per_symbol",
    "encoded_bits",
    "compressed_bytes_without_metadata",
    "archive_payload_bytes",
    "metadata_bytes",
    "total_bytes",
    "compression_ratio_without_metadata",
    "compression_ratio_with_metadata",
]

comparison[columns]

,block_size,alphabet_size,entropy_per_symbol,avg_len_per_symbol,encoded_bits,compressed_bytes_without_metadata,archive_payload_bytes,metadata_bytes,total_bytes,compression_ratio_without_metadata,compression_ratio_with_metadata
0,1,97,4.506665,4.542398,8794409,1099302,1099302,2069,1101371,0.554373,0.555417
1,2,1583,4.023348,4.036578,7815105,976889,976889,40114,1017003,0.492641,0.512870


## 5. Несколько самых частых блоков

In [6]:
for result in results:
    print()
    print(f"Блоки размера {result['block_size']}")
    rows = []

    for block, freq in result["frequencies"].most_common(15):
        rows.append({
            "block": repr(block),
            "frequency": freq,
            "code": result["codes"][block],
            "code_length": len(result["codes"][block]),
        })

    display(pd.DataFrame(rows))


Блоки размера 1


,block,frequency,code,code_length
0,' ',320207,110,3
1,'e',177897,000,3
2,'t',134194,1011,4
3,'o',118338,1000,4
4,'a',117827,0111,4
5,'n',99863,0101,4
6,'h',95335,0100,4
7,'i',93208,0011,4
8,'s',88093,11111,5
9,'r',78542,11101,5



Блоки размера 2


,block,frequency,code,code_length
0,'e ',26208,01010,5
1,' t',21207,111110,6
2,'th',19383,111001,6
3,'he',19146,110111,6
4,'t ',17117,101111,6
5,' a',16934,101101,6
6,'d ',16192,101000,6
7,"', '",13469,011000,6
8,' h',13331,010111,6
9,'s ',12858,010001,6


## Вывод

Кодирование блоками из двух символов обычно дает среднюю длину на один исходный символ ближе к энтропии, потому что алгоритм учитывает зависимости между соседними символами. При этом алфавит блоков становится больше, поэтому таблица кодов тоже растет.